In [ ]:
import pandas as pd
from load import MergedExposureDataset

In [ ]:
df = MergedExposureDataset(reload=True).getMergedExposure()

In [ ]:
from load import MonitorsDataset
mdf = MonitorsDataset().getMonitorsDataset()

In [ ]:
mdf.groupby('location_label').size().reset_index(name='monitor_counts')

In [ ]:
df = df.reset_index()

In [ ]:
df.groupby("classification").agg({
    'location_label':'count',
    'avg_yrly_smoke_pm_gt_5': ['sum','mean', 'std', ],  
    'avg_yrly_smoke_pm': ['mean', 'std',],
    'avg_yrly_pm25_gt_5': ['sum', 'mean', 'std',],
    'avg_yrly_pm25': ['mean', 'std',],
    'distance': ['mean', 'std', ],  
    'monitor_count': ['count', 'sum'], 
    'monitors_sqkm': ['mean', 'std',],
    'popw_avg_yrly_smoke_pm': ['sum', 'std',],
}).round(2).T

In [ ]:
def get_exposure_quantile(df, col, quantiles=3):
    df['Quantile'] = pd.qcut(df[col], quantiles, labels=False)
    df['Quantile_Label'] = pd.qcut(df[col], quantiles, labels=[f'Q{i+1}' for i in range(quantiles)])

    return df

def format_numbers(val):
    if isinstance(val, (int, float)):
        return f"{val:,.2f}"
    return val

In [ ]:
gdf = df

In [ ]:
gdf.columns

In [ ]:
df["avg_yrly_smoke_pm"].sum()

In [ ]:
df["popw_avg_yrly_smoke_pm"].sum()

In [ ]:
quantiles = 4

col = 'avg_yrly_smoke_pm'
res = get_exposure_quantile(gdf, col, quantiles)

grouped = res.groupby(["classification", "Quantile_Label"])

# 1. Weighted Distance Metric: sum of (c * d)
weighted_distance_metric = grouped.apply(lambda x: (x[col] * x['distance']).sum() / len(x))

# 2. Inverse Distance Weighted Concentration: sum of (c / d)
inverse_distance_metric = grouped.apply(lambda x: (x[col] / x['distance']).sum()/ len(x))

# 4. Concentration Gradient Sensitivity: sum of (c / d^2)
concentration_gradient_metric = grouped.apply(lambda x: (x[col] / (x['distance']**2)).sum()/ len(x))

res = grouped.agg({
    col: ['mean', 'std'],  
    'popw_avg_yrly_smoke_pm': ['sum', 'std',],
    'monitor_count': ['count', 'sum'], 
    'monitors_sqkm': ['mean', 'std',],
    'distance': ['mean', 'std',],
    })

res['Mean annual smoke PM2.5 (std)'] = res.apply(lambda x: f"{x[col]['mean']:.2f} ({x[col]['std']:.2f})", axis=1)
res['Population weighted mean annual smoke PM2.5 (std)'] = res.apply(
    lambda x: f"{x['popw_avg_yrly_smoke_pm']['sum']:.2f} ({x['popw_avg_yrly_smoke_pm']['std']:.2f})", axis=1)
res['Total monitors']=res['monitor_count']['sum'] 
res['Monitors/SQKM mean (std)'] = res.apply(lambda x: f"{x['monitors_sqkm']['mean']:.2f} ({x['monitors_sqkm']['std']:.2f})", axis=1)
res['Mean distance to monitor (std)'] = res.apply(lambda x: f"{x['distance']['mean']:.2f} ({x['distance']['std']:.2f})", axis=1)

res.drop(columns=['distance', 'monitor_count', 'monitors_sqkm', col, 'popw_avg_yrly_smoke_pm'], inplace=True)

# A smaller value indicates better monitor placement because it means that monitors are closer to locations with higher concentrations.
res['Mean weighted distance to monitor'] = weighted_distance_metric

# This metric gives higher scores when high concentrations are located closer to the monitor. 
# A larger value indicates better monitor placement because it gives more weight to high concentrations at locations close to the monitor.
res['Mean inverse distance'] = inverse_distance_metric

# This metric assumes that concentration decreases rapidly with distance (like in diffusion-based scenarios). 
# A higher score would indicate that the monitor is well-placed near higher concentrations.
res['Mean concentration gradient'] = concentration_gradient_metric

df_formatted = res.applymap(format_numbers)

df_formatted.T

In [ ]:
grouped = gdf.groupby(["STATE_ABBR","classification"])

# 1. Weighted Distance Metric: sum of (c * d)
weighted_distance_metric = grouped.apply(lambda x: (x[col] * x['distance']).sum() / len(x))

# 2. Inverse Distance Weighted Concentration: sum of (c / d)
inverse_distance_metric = grouped.apply(lambda x: (x[col] / x['distance']).sum()/ len(x))

# 4. Concentration Gradient Sensitivity: sum of (c / d^2)
concentration_gradient_metric = grouped.apply(lambda x: (x[col] / (x['distance']**2)).sum()/ len(x))

In [ ]:
concentration_gradient_metric=concentration_gradient_metric.round(2)
inverse_distance_metric =inverse_distance_metric.round(2)
weighted_distance_metric= weighted_distance_metric.round(2)

In [ ]:
gdf.columns

In [ ]:
gdf["monitor_count"]=gdf["monitor_count"].fillna(0)
gdf["monitor_count"]=gdf["monitor_count"].astype(int)
gdf["monitor_count"]

In [ ]:
res = gdf.groupby(["STATE_ABBR","classification"]).agg({
    'classification': 'size',
    col: ['mean', 'std'],  
    'popw_avg_yrly_smoke_pm': ['sum', 'std',],
    'avg_yrly_smoke_pm_gt_5': ['sum',],
    'monitor_count': ['count', 'sum'], 
    'monitors_sqkm': ['mean', 'std',],
    'distance': ['mean', 'std',],
})

res

In [ ]:
with_monitor = gdf.groupby(["STATE_ABBR","classification"])['monitor_count'].apply(lambda x: (x != 0).sum())
with_monitor.name = 'locations_with_monitor'
with_monitor

## Main Table

In [ ]:
gdf['smoke_person_days'] = gdf['POPULATION'] * gdf['sum_smoke_pm_gt_5']
gdf['pm25_person_days'] = gdf['POPULATION'] * gdf['sum_pm25_gt_35']

In [ ]:
col = 'avg_yrly_smoke_pm'

res = gdf.groupby(["STATE_ABBR","classification"]).agg({
    'classification': 'size',
    col: ['mean', 'std'],  
    'popw_avg_yrly_smoke_pm': ['sum', 'std',],
    'smoke_person_days': ['sum'],
    'pm25_person_days': ['sum'],
    'avg_yrly_smoke_pm_gt_5': ['sum',],
    'monitor_count': ['count', 'sum'], 
    'monitors_sqkm': ['mean', 'std',],
    'distance': ['mean', 'std',],
})


res['Number of census tracts']=res['classification']['size'] 
res['Number of census tracts with a monitor site']=with_monitor
res['Number of USPEA monitors']=res['monitor_count']['sum'] 
res['Number of monitors per census tract'] = (res['Number of USPEA monitors']/res['Number of census tracts']).round(2)

res['Mean distance to monitor (std)'] = res.apply(lambda x: f"{x['distance']['mean']:.1f} ({x['distance']['std']:.1f})", axis=1)
res['Monitors/SQKM mean (std) (multiplied by 1000)'] = res.apply(lambda x: f"{x['monitors_sqkm']['mean']:.1f} ({x['monitors_sqkm']['std']:.1f})", axis=1)
res['Average annual mean wildfire smoke PM2.5 concentration, μg/m3 (std)'] = res.apply(lambda x: f"{x[col]['mean']:.1f} ({x[col]['std']:.1f})", axis=1)
res['Population weighted mean annual smoke PM2.5 (std)'] = res.apply(
    lambda x: f"{x['popw_avg_yrly_smoke_pm']['sum']:.2f} ({x['popw_avg_yrly_smoke_pm']['std']:.2f})", axis=1)

res['Total smoke days (PM2.5 > 5 μg/m3)'] = res['avg_yrly_smoke_pm_gt_5']['sum'].astype(int)

res["Average smoke days per year per census tract"]= res['Total smoke days (PM2.5 > 5 μg/m3)']/res['Number of census tracts']
res["Average smoke days per year per census tract"] = res["Average smoke days per year per census tract"].round(1)

res.drop(columns=['distance', 'monitor_count', 'monitors_sqkm','classification', col, 'avg_yrly_smoke_pm_gt_5','popw_avg_yrly_smoke_pm'], inplace=True)

# A smaller value indicates better monitor placement because it means that monitors are closer to locations with higher concentrations.
res['Mean exposure--weighted distance to monitor'] = weighted_distance_metric

res['smoke_person_days'] = res['smoke_person_days']/1000000
res['smoke_person_days'] = res['smoke_person_days'].round(1)
res['pm25_person_days'] = res['pm25_person_days']/1000000
res['pm25_person_days'] = res['pm25_person_days'].round(1)

res = res.rename(columns={
    'pm25_person_days': 'Total ambient PM2.5 person-days (in millions)',
    'smoke_person_days': 'Total Wildfire smoke PM2.5 person-days (in millions)',
})  

res.columns = res.columns.droplevel(1)

In [ ]:
# Sort by avg_yrly_smoke_pm_gt_5 (sum)
sort_col = "Total smoke days (PM2.5 > 5 μg/m3)"
# 1. Pull rural values per state
rural_vals = res.xs("Rural", level="classification")[sort_col]
# 2. Broadcast rural values back to all classifications in each state
res["rural_sort_key"] = (
    res.index.get_level_values("STATE_ABBR").map(rural_vals)
)
# 3. Sort by the rural-based key
res = res.reset_index().rename(columns={"STATE_ABBR": "STATE", "classification": "classification"})
res_sorted = res.sort_values(["rural_sort_key","STATE",], ascending=[False,True])
# res_sorted = res_sorted.drop(columns=["rural_sort_key"])
res = res_sorted


In [ ]:
res.drop(columns=["rural_sort_key"], inplace=True)

In [ ]:
res = res.groupby(["STATE","classification"], sort=False).first()

In [ ]:
res = res[[
       'Number of census tracts',
       'Number of census tracts with a monitor site',
       'Number of USPEA monitors', 'Number of monitors per census tract',
       'Mean distance to monitor (std)',
       'Monitors/SQKM mean (std) (multiplied by 1000)',
       'Average annual mean wildfire smoke PM2.5 concentration, μg/m3 (std)',
       'Average smoke days per year per census tract',
       'Total smoke days (PM2.5 > 5 μg/m3)',
       'Mean exposure--weighted distance to monitor',
       'Total Wildfire smoke PM2.5 person-days (in millions)',
       'Total ambient PM2.5 person-days (in millions)',
       'Population weighted mean annual smoke PM2.5 (std)']]

In [ ]:
upper_map = {
    # "STATE": "Census tract descriptives",
    # "classification": "Census tract descriptives",
    "Number of census tracts": "Census tract descriptives",
    "Number of census tracts with a monitor site": "Census tract descriptives",

    "Number of USPEA monitors": "US EPA monitor information",
    "Number of monitors per census tract": "US EPA monitor information",
    "Mean distance to monitor (std)": "US EPA monitor information",
    "Monitors/SQKM mean (std) (multiplied by 1000)": "US EPA monitor information",

    "Mean exposure--weighted distance to monitor": "Annual PM2.5 exposure",
    "Average annual mean wildfire smoke PM2.5 concentration, μg/m3 (std)": "Annual PM2.5 exposure",
    "Total smoke days (PM2.5 > 5 μg/m3)": "Annual PM2.5 exposure",
    "Average smoke days per year per census tract": "Annual PM2.5 exposure",

    "Total Wildfire smoke PM2.5 person-days (in millions)": "Population burden of PM2.5",
    "Total ambient PM2.5 person-days (in millions)": "Population burden of PM2.5",
    "Population weighted mean annual smoke PM2.5 (std)": "Population burden of PM2.5"
}

# Build MultiIndex columns
res.columns = pd.MultiIndex.from_tuples(
    [(upper_map[col], col) for col in res.columns]
)

In [ ]:
res

## Urban-Rural Ratios

In [ ]:
urban_ratio = df.groupby(["STATE_ABBR", "classification"])["POPULATION"].sum().unstack()

In [ ]:
urban_ratio['ratio'] = urban_ratio['Urban'] / urban_ratio['Rural']
urban_ratio['ratio'] = urban_ratio['ratio'].round(2)

In [ ]:
urban_ratio

In [ ]:
urban_ratio = df.groupby(["STATE_ABBR", "classification"])['location_label'].count().unstack()

In [ ]:
urban_ratio['ratio'] = urban_ratio['Urban'] / urban_ratio['Rural']
urban_ratio['ratio'] = urban_ratio['ratio'].round(1)
urban_ratio

In [ ]:
urban_ratio = df[df['monitor_count'] >0].groupby(["STATE_ABBR", "classification"])['location_label'].count().unstack()
urban_ratio['ratio'] = urban_ratio['Urban'] / urban_ratio['Rural']
urban_ratio['ratio'] = urban_ratio['ratio'].round(1)
urban_ratio